# Problem 1 — Recurrent models

This notebook imports the tested implementations in `solution.py`. The quick path verifies the generated-data contract, scratch cells, and parameter counts. Set `RUN_FULL_EXPERIMENTS = True` to download UCI HAR and run the complete seed/model/sequence-length studies; run that path on a GPU (the brief specifies a Colab T4). Results are appended to the root `results.csv` after each run. The individual ablation for Task 1.5(c) isolates gradient clipping on the LSTM adding task at T=200.

The assignment excerpt provides the 1.5 rubric but no more specific definition of the individual ablation. Gradient clipping is chosen because it directly tests the gradient-stability intervention used in the long-sequence experiments.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'problem1':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from problem1.solution import *
RUN_FULL_EXPERIMENTS = False  # Switch to True on a GPU for the full marked experiments.

## 1.1 Adding data contract

Every sequence contains exactly two marked values, one in each half, and its target is their sum.

In [2]:
test_make_adding()
print('Adding-problem generator assertions passed.')

Adding-problem generator assertions passed.


## 1.2 Scratch recurrent cells

Each custom cell uses only matrix operations and nonlinearities. The comparison copies the corresponding PyTorch parameters and unrolls over the same sequence.

In [3]:
cell_errors = test_cells_match_torch()
cell_counts = hand_parameter_counts(9, 64)
assert cell_counts == {'RNN_hand': 4736, 'LSTM_hand': 18944, 'GRU_hand': 14208,
                       'RNN_torch': 4800, 'LSTM_torch': 19200, 'GRU_torch': 14400}
print('Maximum absolute cell errors:', cell_errors)
print('Hand counts and PyTorch counts:', cell_counts)
print('The additional PyTorch parameters are the second bias vector (one per gate).')

Maximum absolute cell errors: {'RNN': 0.0, 'LSTM': 5.960464477539063e-08, 'GRU': 1.1920928955078125e-07}
Hand counts and PyTorch counts: {'RNN_hand': 4736, 'LSTM_hand': 18944, 'GRU_hand': 14208, 'RNN_torch': 4800, 'LSTM_torch': 19200, 'GRU_torch': 14400}
The additional PyTorch parameters are the second bias vector (one per gate).


## Full experiments (GPU recommended)

All three HAR models use hidden size 64, Adam at 1e-3, batch size 128, and 8 epochs for seeds 0–2. The adding sweep uses 10,000/1,000/1,000 train/validation/test sequences, fixed data seeds 0/1/2, 20 epochs, and 3 model seeds. Longer sweeps are compute intensive.

In [4]:
if RUN_FULL_EXPERIMENTS:
    har_rows = []
    best_histories = None
    for experiment_seed in (0, 1, 2):
        train_split, val_split, test_split, held_out = load_har(experiment_seed)
        plot_har_class_balance(train_split[1], test_split[1])
        result, histories = train_har(experiment_seed, epochs=8, batch_size=128, hidden_size=64)
        har_rows.append(result)
        best_histories = histories
    har_results = pd.concat(har_rows, ignore_index=True)
    print(har_results.groupby('model')[['test_acc', 'macro_f1', 'params', 'sec_per_epoch']].agg(['mean', 'std']))
    plot_har_confusion(best_histories)

    adding_results, adding_histories = run_adding_sweep(epochs=20, hidden_size=64)
    solved = plot_adding_results(adding_results, adding_histories)
    print('Solved runs (test MSE < 0.05):\n', solved)
    plot_gradient_norm(seed=0, T=200, hidden_size=64)
    hidden_sweep = run_hidden_size_sweep(epochs=8)
    clipping_ablation = run_clipping_ablation(epochs=20)
    display(hidden_sweep, clipping_ablation)
    print('Figures saved to problem1/figures; run rows saved to results.csv.')
else:
    print('Full experiments skipped. Set RUN_FULL_EXPERIMENTS=True on a GPU to execute them.')

Full experiments skipped. Set RUN_FULL_EXPERIMENTS=True on a GPU to execute them.


## Interpreting the evidence

Use the per-seed result rows and mean ± standard deviation rather than a single run. Compare adding-task test MSE with the constant-prediction baseline near 0.167; a run is solved only below 0.05. The clipping ablation is controlled by fixing model, seed, length, data and epoch budget. For a wearable with limited compute, justify a model using measured accuracy, parameter count and epoch time rather than accuracy alone.